# SharkByte Colab: 33,554,432-state scaling run

This notebook runs one new CPU experiment. It does **not** rerun the integrated suite or the completed 16,777,216-state benchmark.

It starts from the archived 40-qubit circuit and its verified 16,777,216-state receipt, then appends one Hadamard on qubit 1. A local component-level probe predicts 33,554,432 nonzero amplitudes. Cirq independently simulates the small disconnected components and the worker compares every populated SharkByte amplitude against their tensor product. This is a sparse-support scaling check, **not** evidence of global 40-qubit entanglement or a dense 40-qubit simulation.

Use a **CPU** runtime. The run refuses to start below 9 GiB available RAM. The worker has a 7 GiB address-space limit, a 6.5 GiB sampled RSS stop, a 300-second simulator limit, a 480-second CPU-time limit, and parent-side timeout/interruption cleanup. If a guard refuses or stops it, the notebook records that status; do not describe it as a pass.

The receipt contains summary metrics and provenance only; it does not export millions of amplitudes.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import platform
import signal
import subprocess
import sys
import tempfile
import time
import zipfile

GIB = 1024 ** 3
REPO_COMMIT = "151ed72caa4d0a1424e1660710e9a426b7223af4"
PARENT_RECEIPT_SHA256 = "D3560811FF8885C50AEDAF2B8DFF0AF7F70B66804182A9F89BC2543762493DA6".lower()
EXPECTED_SOURCE_SHA256 = "A24D8F389850E4A298F611D45AB4C5E953EAD59C153777071916ECEA73CD5316".lower()
TARGET_STATES = 1 << 25
MIN_AVAILABLE_BYTES = 9 * GIB
WORKER_AS_LIMIT_BYTES = 7 * GIB
WORKER_RSS_LIMIT_BYTES = int(6.5 * GIB)
RUN_DIR = Path(tempfile.mkdtemp(prefix="sharkbyte-32m-", dir="/content"))
REPO = RUN_DIR / "SharkByte"
RESULTS = RUN_DIR / "results"
RESULTS.mkdir()
PACKAGES = RUN_DIR / "python-packages"
PACKAGES.mkdir()
WORKER = RUN_DIR / "run_32m.py"
RECEIPT = RESULTS / "sharkbyte-40q-33554432-colab.json"
LOG = RESULTS / "worker.log"
PROGRESS = RESULTS / "progress.jsonl"

clone = subprocess.run(
    ["git", "clone", "--quiet", "https://github.com/reggaesharkk/SharkByte.git", str(REPO)],
    capture_output=True, text=True
)
if clone.returncode:
    raise RuntimeError(clone.stderr or "GitHub clone failed")
subprocess.run(["git", "checkout", "--quiet", "--detach", REPO_COMMIT], cwd=REPO, check=True)
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if actual_commit != REPO_COMMIT:
    raise RuntimeError(f"Expected {REPO_COMMIT}, got {actual_commit}")

WORKER_PYTHON = Path(sys.executable)
WORKER_ENV = dict(os.environ, MPLBACKEND="Agg")
existing_pythonpath = WORKER_ENV.get("PYTHONPATH", "")
WORKER_ENV["PYTHONPATH"] = str(PACKAGES) + (os.pathsep + existing_pythonpath if existing_pythonpath else "")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
     "--target", str(PACKAGES), "numpy==2.5.4", "cirq-core==1.6.1"],
    check=True
)
env_info = subprocess.check_output(
    [str(WORKER_PYTHON), "-c",
     "import sys, numpy, cirq; print(sys.version.split()[0], numpy.__version__, cirq.__version__)"],
    cwd=REPO, env=WORKER_ENV, text=True
).strip()
print("Pinned source:", actual_commit)
print("Python / NumPy / Cirq:", env_info)
print("Target active states:", f"{TARGET_STATES:,}")
print("CPU runtime:", platform.platform())


In [ ]:
WORKER_SOURCE = r'''import argparse
import hashlib
from importlib.metadata import version
from itertools import islice
import json
import math
from pathlib import Path
import platform
import subprocess
import sys
from time import perf_counter

import cirq
import numpy as np

parser = argparse.ArgumentParser()
parser.add_argument("--repo", required=True)
parser.add_argument("--output", required=True)
parser.add_argument("--progress", required=True)
parser.add_argument("--worker-sha256", required=True)
parser.add_argument("--address-space-limit", type=int, required=True)
args = parser.parse_args()

repo = Path(args.repo).resolve()
source_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=repo, text=True
).strip()
sys.path.insert(0, str(repo / "tools"))
sys.path.insert(0, str(repo / "benchmarks" / "sb_quantum_benchmark"))
from frontier30 import load_circuit
from sparse_backend import Circuit, EvolutionStats, evolve

STATES = 1 << 25
TOLERANCE = 1e-12
REFERENCE_ZERO = 1e-14
BATCH = 1 << 16
source = repo / "examples" / "frontier_sparse_40_131072.sb"
parent_path = repo / "benchmarks" / "sb_quantum_benchmark" / "scaling-16777216-ubuntu-wsl.json"
parent_raw = parent_path.read_bytes()
parent_hash = hashlib.sha256(parent_raw).hexdigest()
if parent_hash != "d3560811ff8885c50aedaf2b8dff0af7f70b66804182a9f89bc2543762493da6":
    raise ValueError("Historical 16M receipt SHA-256 mismatch")
parent = json.loads(parent_raw)["result"]
base, source_hash = load_circuit(source)
if source_hash != "a24d8f389850e4a298f611d45ab4c5e953ead59c153777071916ecea73cd5316":
    raise ValueError("Frozen circuit source SHA-256 mismatch")
if (parent["status"], parent["qubits"], parent["gates"], parent["final_active_states"],
    parent["source_sha256"]) != ("PASS", 40, 56, 1 << 24, source_hash):
    raise ValueError("Historical 16M receipt does not match the expected baseline")
added = [tuple(gate) for gate in parent["added_gates"]] + [("h", 1)]
circuit = Circuit(base.qubits, base.gates + tuple(added), "support_scaling_33554432")
if circuit.qubits != 40 or len(circuit.gates) != 57:
    raise ValueError("Unexpected target circuit dimensions")

identity = {
    "format": "sharkbyte-colab-32m-circuit-v1",
    "source_sha256": source_hash,
    "parent_receipt_sha256": parent_hash,
    "added_gates": added,
    "qubits": circuit.qubits,
    "gate_count": len(circuit.gates),
}
circuit_sha256 = hashlib.sha256(
    json.dumps(identity, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()

def cirq_state(width, gates):
    qubits = cirq.LineQubit.range(width)
    reference = cirq.Circuit()
    single = {"h": cirq.H, "x": cirq.X, "z": cirq.Z}
    rotations = {"rx": cirq.rx, "ry": cirq.ry, "rz": cirq.rz}
    for op, *values in gates:
        if op in single:
            operation = single[op](qubits[values[0]])
        elif op == "cx":
            operation = cirq.CNOT(qubits[values[0]], qubits[values[1]])
        elif op == "cp":
            control, target, angle = values
            operation = cirq.CZPowGate(exponent=angle / math.pi)(
                qubits[control], qubits[target]
            )
        else:
            target, angle = values
            operation = rotations[op](angle)(qubits[target])
        reference.append(operation, strategy=cirq.InsertStrategy.NEW)
    return cirq.Simulator(dtype=np.complex128).simulate(
        reference, qubit_order=qubits[::-1]
    ).final_state_vector

def factorize(program):
    groups = [{q} for q in range(program.qubits)]
    for op, *values in program.gates:
        if op not in ("cx", "cp"):
            continue
        control, target = values[:2]
        left = next(group for group in groups if control in group)
        right = next(group for group in groups if target in group)
        if left is not right:
            left.update(right)
            groups.remove(right)
    for group in groups:
        indices = tuple(sorted(group))
        local = {q: i for i, q in enumerate(indices)}
        local_gates = []
        for op, *values in program.gates:
            if values[0] not in local:
                continue
            if op in ("cx", "cp"):
                local_gates.append((op, local[values[0]], local[values[1]], *values[2:]))
            else:
                local_gates.append((op, local[values[0]], *values[1:]))
        yield indices, cirq_state(len(indices), local_gates)

factors = list(factorize(circuit))
counts = [int(np.count_nonzero(np.abs(values) > REFERENCE_ZERO))
          for _, values in factors]
norms = [math.fsum(abs(complex(value)) ** 2 for value in values)
         for _, values in factors]
reference_count = math.prod(counts)
reference_norm = math.prod(norms)
if reference_count != STATES:
    raise ValueError(f"Cirq component support predicts {reference_count}, expected {STATES}")
if any(not math.isfinite(norm) or norm <= 0 for norm in norms):
    raise ValueError("Invalid Cirq component norm")

def report_progress(phase, gate_index, gate_count, active_states):
    row = {"phase": phase, "gate_index": gate_index, "gate_count": gate_count,
           "active_states": active_states}
    with Path(args.progress).open("a", encoding="utf-8", newline="\n") as stream:
        stream.write(json.dumps(row) + "\n")
        stream.flush()

report_progress("evolving", 0, len(circuit.gates), 1)
stats = EvolutionStats()
started = perf_counter()
state = evolve(
    circuit, support_cap=STATES, memory_budget_bytes=STATES * 512,
    max_seconds=300, stats=stats,
    progress_callback=lambda done, total, active: report_progress(
        "evolving", done, total, active
    )
)
evolution_seconds = perf_counter() - started
report_progress("comparing", 0, len(state), len(state))
peak_error = 0.0
peak_probability_error = 0.0
minimum_reference = math.inf
entries = iter(state.items())
compared = 0
while batch := list(islice(entries, BATCH)):
    basis = np.fromiter((key for key, _ in batch), dtype=np.uint64, count=len(batch))
    actual = np.fromiter((amp for _, amp in batch), dtype=np.complex128, count=len(batch))
    expected = np.ones(len(batch), dtype=np.complex128)
    for qubits, amplitudes in factors:
        local = np.zeros(len(batch), dtype=np.uint64)
        for index, q in enumerate(qubits):
            local |= ((basis >> q) & 1) << index
        expected *= amplitudes[local]
    peak_error = max(peak_error, float(np.max(np.abs(actual - expected))))
    peak_probability_error = max(
        peak_probability_error,
        float(np.max(np.abs(np.abs(actual) ** 2 - np.abs(expected) ** 2)))
    )
    minimum_reference = min(minimum_reference, float(np.min(np.abs(expected))))
    compared += len(batch)
comparison_seconds = perf_counter() - started - evolution_seconds
probability_sum = math.fsum(abs(amplitude) ** 2 for amplitude in state.values())
passed = (
    len(state) == stats.peak_support == reference_count == compared == STATES
    and minimum_reference > REFERENCE_ZERO
    and abs(reference_norm - 1.0) <= TOLERANCE
    and abs(probability_sum - 1.0) <= TOLERANCE
    and peak_error <= TOLERANCE
    and peak_probability_error <= TOLERANCE
)
report = {
    "status": "PASS" if passed else "FAIL",
    "claim_scope": "Exact finite sparse evolution; every populated amplitude compared with a tensor product of independent Cirq component simulations. The circuit has disconnected components.",
    "execution_id": circuit_sha256[:32],
    "circuit_sha256": circuit_sha256,
    "source_commit": source_commit,
    "source_sha256": source_hash,
    "parent_16m_receipt_sha256": parent_hash,
    "worker_sha256": args.worker_sha256,
    "qubits": circuit.qubits,
    "gate_count": len(circuit.gates),
    "added_gates_after_16m_baseline": [["h", 1]],
    "all_added_gates": added,
    "target_active_states": STATES,
    "components": [
        {"qubits": list(qubits),
         "active_states": int(count),
         "norm": norm}
        for (qubits, _), count, norm in zip(factors, counts, norms)
    ],
    "reference_active_states": reference_count,
    "sharkbyte_peak_active_states": stats.peak_support,
    "compared_amplitudes": compared,
    "reference_norm": reference_norm,
    "probability_sum": probability_sum,
    "max_complex_amplitude_error": peak_error,
    "max_probability_error": peak_probability_error,
    "minimum_reference_amplitude_magnitude": minimum_reference,
    "amplitude_tolerance": TOLERANCE,
    "component_zero_threshold": REFERENCE_ZERO,
    "sharkbyte_evolution_seconds": evolution_seconds,
    "cirq_comparison_seconds": comparison_seconds,
    "environment": {
        "platform": platform.platform(),
        "architecture": platform.machine(),
        "python": platform.python_version(),
        "cirq_core": version("cirq-core"),
        "numpy": np.__version__,
        "worker_address_space_limit_bytes": args.address_space_limit,
    },
}
with Path(args.output).open("x", encoding="utf-8", newline="\n") as stream:
    json.dump(report, stream, indent=2, allow_nan=False)
    stream.write("\n")
print(json.dumps({"status": report["status"], "compared_amplitudes": compared,
                  "max_error": peak_error, "probability_sum": probability_sum}))
if not passed:
    raise SystemExit(1)'''
print("Embedded worker SHA-256:", hashlib.sha256(WORKER_SOURCE.encode("utf-8")).hexdigest())
print("The worker checks the archived 16M receipt before building the 57-gate target.")


In [ ]:
def available_memory_bytes():
    meminfo = Path("/proc/meminfo").read_text(encoding="utf-8")
    available = int(next(line.split()[1] for line in meminfo.splitlines()
                         if line.startswith("MemAvailable:"))) * 1024
    for limit_path, usage_path in (
        ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
        ("/sys/fs/cgroup/memory/memory.limit_in_bytes",
         "/sys/fs/cgroup/memory/memory.usage_in_bytes"),
    ):
        try:
            limit = Path(limit_path).read_text(encoding="utf-8").strip()
            if limit.isdecimal():
                usage = int(Path(usage_path).read_text(encoding="utf-8"))
                available = min(available, max(0, int(limit) - usage))
        except FileNotFoundError:
            continue
    return available

def worker_rss_bytes(pid):
    try:
        status = Path(f"/proc/{pid}/status").read_text(encoding="utf-8")
        line = next(row for row in status.splitlines() if row.startswith("VmRSS:"))
        return int(line.split()[1]) * 1024
    except (FileNotFoundError, StopIteration, ProcessLookupError):
        return 0

def terminate_worker(proc):
    if proc.poll() is not None:
        return
    try:
        os.killpg(proc.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        proc.wait()

available_start = available_memory_bytes()
print(f"Available RAM before run: {available_start / GIB:.2f} GiB")
print(f"Worker address-space cap: {WORKER_AS_LIMIT_BYTES / GIB:.1f} GiB")
if available_start < MIN_AVAILABLE_BYTES:
    outcome = {
        "status": "REFUSED",
        "reason": "Available RAM is below the 9 GiB preflight threshold.",
        "available_bytes": available_start,
        "required_available_bytes": MIN_AVAILABLE_BYTES,
    }
    (RESULTS / "run-status.json").write_text(json.dumps(outcome, indent=2) + "\n")
    print(outcome)
else:
    WORKER.write_text(WORKER_SOURCE, encoding="utf-8", newline="\n")
    worker_sha256 = hashlib.sha256(WORKER.read_bytes()).hexdigest()

    def set_worker_limits():
        import resource
        resource.setrlimit(resource.RLIMIT_AS,
                           (WORKER_AS_LIMIT_BYTES, WORKER_AS_LIMIT_BYTES))
        resource.setrlimit(resource.RLIMIT_CPU, (480, 480))
        resource.setrlimit(resource.RLIMIT_CORE, (0, 0))

    outcome = None
    peak_rss = 0
    started = time.monotonic()
    with LOG.open("w", encoding="utf-8", newline="\n") as log_stream:
        proc = subprocess.Popen(
            [str(WORKER_PYTHON), str(WORKER), "--repo", str(REPO),
             "--output", str(RECEIPT), "--progress", str(PROGRESS),
             "--worker-sha256", worker_sha256,
             "--address-space-limit", str(WORKER_AS_LIMIT_BYTES)],
            cwd=REPO,
            env=WORKER_ENV,
            stdout=log_stream,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            preexec_fn=set_worker_limits,
        )
        try:
            while proc.poll() is None:
                peak_rss = max(peak_rss, worker_rss_bytes(proc.pid))
                available_now = available_memory_bytes()
                elapsed = time.monotonic() - started
                if peak_rss > WORKER_RSS_LIMIT_BYTES:
                    outcome = {"status": "RESOURCE_STOP",
                               "reason": "Worker RSS crossed the 6.5 GiB stop threshold."}
                    terminate_worker(proc)
                    break
                if available_now < 2 * GIB:
                    outcome = {"status": "RESOURCE_STOP",
                               "reason": "Available RAM fell below the 2 GiB safety floor."}
                    terminate_worker(proc)
                    break
                if elapsed > 600:
                    outcome = {"status": "TIMEOUT",
                               "reason": "Parent timeout reached 600 seconds."}
                    terminate_worker(proc)
                    break
                time.sleep(1)
        except KeyboardInterrupt:
            terminate_worker(proc)
            outcome = {"status": "CANCELLED",
                       "reason": "Notebook interrupt received; worker process group was terminated."}
        return_code = proc.poll()
    elapsed_total = time.monotonic() - started

    if outcome is None:
        if return_code == 0 and RECEIPT.exists():
            record = json.loads(RECEIPT.read_bytes())
            record["colab_measurement"] = {
                "available_ram_at_start_bytes": available_start,
                "sampled_peak_worker_rss_bytes": peak_rss,
                "parent_elapsed_seconds": elapsed_total,
                "worker_address_space_limit_bytes": WORKER_AS_LIMIT_BYTES,
                "worker_sampled_rss_stop_bytes": WORKER_RSS_LIMIT_BYTES,
            }
            RECEIPT.write_text(json.dumps(record, indent=2, allow_nan=False) + "\n",
                               encoding="utf-8", newline="\n")
            receipt_sha = hashlib.sha256(RECEIPT.read_bytes()).hexdigest()
            RECEIPT.with_suffix(".sha256").write_text(
                f"{receipt_sha}  {RECEIPT.name}\n", encoding="utf-8", newline="\n"
            )
            outcome = {"status": record["status"], "receipt_sha256": receipt_sha,
                       "sampled_peak_worker_rss_bytes": peak_rss}
        else:
            outcome = {
                "status": outcome["status"] if outcome else "WORKER_FAILED",
                "reason": outcome["reason"] if outcome else (
                    f"Worker exited with code {return_code} without a completed receipt."
                ),
                "return_code": return_code,
                "sampled_peak_worker_rss_bytes": peak_rss,
            }

    (RESULTS / "run-status.json").write_text(json.dumps(outcome, indent=2) + "\n",
                                              encoding="utf-8", newline="\n")
    print("Run outcome:", json.dumps(outcome))
    if LOG.exists():
        tail = LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-30:]
        print("\n".join(tail))
    if PROGRESS.exists():
        print("Last progress:", PROGRESS.read_text(encoding="utf-8").splitlines()[-1])


In [ ]:
# Download the compact receipt and run log/status bundle.
archive = RUN_DIR / "sharkbyte-colab-32m-results.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RESULTS.iterdir()):
        if path.is_file():
            bundle.write(path, arcname=path.name)
print("Evidence bundle:", archive)
from google.colab import files
files.download(str(archive))